# Jam Transformer — 실제 데이터 학습 검증

**목적:** Google Drive에 올려둔 번들(`jam_tx_bundle.tgz`)을 Colab 로컬에 복사·압축해제 후,
실제 데이터로 학습하여 **loss가 내려가는지** 확인합니다.

전체 epoch 학습은 4070 Ti Super / RunPod 에서 진행 — 여기서는 **1 epoch**만 돌립니다.

| 단계 | 내용 |
|---|---|
| 0 | GPU 확인 |
| 1 | GDrive 마운트 + 번들 복사·해제 |
| 2 | 의존성 설치 |
| 3 | 데이터 무결성 검증 |
| 4 | 학습 (1 epoch) + loss 시각화 |
| 5 | 추론 — 실제 멜로디 → 반주 생성 |

> **번들 준비 (로컬 PC):**
> ```powershell
> # 소스 코드 + 전체 데이터 한 번에 번들링 (~800MB~1.2GB 압축)
> docker compose run --rm train python scripts/package_for_server.py --compress gz
> ```
> `jam_tx_bundle.tgz` 를 Google Drive에 업로드하세요.
>
> **런타임 설정:** `런타임 > 런타임 유형 변경 > T4 GPU` 선택 후 실행하세요.

## 0. GPU 확인

In [ ]:
import subprocess, sys

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode != 0 or not result.stdout.strip():
    raise RuntimeError(
        "GPU가 없습니다. [런타임 > 런타임 유형 변경 > T4 GPU] 로 변경 후 재실행하세요."
    )
print("GPU:", result.stdout.strip())

import torch
print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.version.cuda}  |  "
      f"device: {torch.cuda.get_device_name(0)}")

## 1. 경로 설정 (여기만 수정)

Drive에 올린 번들 경로를 맞게 수정하세요.

```
내 드라이브/
  jam_tx_bundle.tgz        # scripts/package_for_server.py 로 생성한 번들
                           # (소스 코드 + data/processed/ 포함, ~4-60 MB)
```

> 번들에 `data/processed/`가 포함되어 있다면 파일 하나만 올리면 됩니다.  
> 데이터를 따로 올린 경우 `DRIVE_DATA_BUNDLE` 도 지정하세요.

In [ ]:
# ============================================================
#  여기만 수정하세요
# ============================================================

# Google Drive 상의 번들 경로 (소스 코드 + data/processed/ 포함)
DRIVE_BUNDLE = '/content/drive/MyDrive/jam_tx_bundle.tgz'

# 번들에 data/processed/가 없는 경우, 데이터 전용 tar 경로를 지정
# 없으면 None 으로 두세요
DRIVE_DATA_BUNDLE = None  # 예: '/content/drive/MyDrive/pop909_processed.tgz'

# 학습 설정
EPOCHS      = 1   # loss 하강 확인용. 풀 학습은 4070 Ti Super / RunPod에서
                  # batch=32 × 8heads × 2560² × 4B = 6.25 GiB → OOM
                  # batch=8 × 8heads × 2560² × 4B = 1.56 GiB → 안전
                     # val_batch=64@seq=2560 → 12.49 GiB 할당 → OOM
                     # 4070 Ti Super(Ampere): Flash Attention 있어서 이 제한 불필요
# batch_size / grad_accum / val_batch_size 는 자동 결정됩니다:
#   Flash Attention 있음 (Ampere+, 4070 Ti Super 등): batch=32, accum=1
#   Flash Attention 없음 (Turing, T4 등):              batch=8,  accum=4 (effective=32)
# --set training.batch_size=N 또는 --set training.accumulate_grad_batches=N 으로 수동 오버라이드 가능
LOG_EVERY   = 10  # 몇 step마다 loss 출력
# ============================================================

## 2. Google Drive 마운트 + 번들 복사 · 해제

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("Drive 마운트 완료")

In [ ]:
import os, shutil, subprocess, time
from pathlib import Path

PROJECT_DIR = Path('/content/project_transformer')

# --- 코드 + 데이터 번들 ---
local_bundle = Path('/content/jam_tx_bundle.tgz')
if not PROJECT_DIR.exists():
    print(f'Drive에서 번들 복사 중... ({DRIVE_BUNDLE})')
    t0 = time.time()
    shutil.copy(DRIVE_BUNDLE, local_bundle)
    print(f'  복사 완료: {local_bundle.stat().st_size/1e6:.1f} MB  '
          f'({time.time()-t0:.1f}s)')

    print('압축 해제 중...')
    t0 = time.time()
    subprocess.run(['tar', 'xzf', str(local_bundle), '-C', '/content/'],
                   check=True)
    print(f'  완료 ({time.time()-t0:.1f}s)  ->  {PROJECT_DIR}')
    local_bundle.unlink()  # 용량 절약
else:
    print(f'{PROJECT_DIR} 이미 존재 — 압축 해제 건너뜀')

# --- 데이터 번들이 따로 있는 경우 ---
data_dir = PROJECT_DIR / 'data' / 'processed'
if DRIVE_DATA_BUNDLE and not data_dir.exists():
    local_data = Path('/content/pop909_processed.tgz')
    print(f'\n데이터 번들 복사 중... ({DRIVE_DATA_BUNDLE})')
    t0 = time.time()
    shutil.copy(DRIVE_DATA_BUNDLE, local_data)
    print(f'  복사 완료: {local_data.stat().st_size/1e6:.1f} MB  '
          f'({time.time()-t0:.1f}s)')
    subprocess.run(['tar', 'xzf', str(local_data), '-C', '/content/'],
                   check=True)
    local_data.unlink()
    print('  데이터 압축 해제 완료')

# 작업 디렉토리 이동
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
print(f'\nCWD: {os.getcwd()}')

## 3. 의존성 설치

PyTorch / CUDA는 Colab에 이미 설치되어 있으므로 재설치하지 않습니다.

In [ ]:
%%bash
pip install -q \
    pytorch-lightning>=2.2 \
    miditoolkit \
    loguru \
    pretty_midi \
    pyyaml \
    wandb

# 패키지 자체 설치 (torch 재설치 없이)
pip install -q -e . --no-build-isolation 2>&1 | tail -3
echo "설치 완료"

## 4. 데이터 무결성 검증

In [ ]:
import json
from pathlib import Path
from jam_transformer.config import load_config
from jam_transformer.tokenizer import build_tokenizer
from jam_transformer.dataset import assert_data_matches_config

cfg = load_config('configs/config.yaml')
tok = build_tokenizer(cfg.tokenizer)

data_dir = Path('data/processed')
shards = sorted(p for p in data_dir.glob('*.pt') if not p.name.startswith('_'))
meta   = json.loads((data_dir / '_dataset_meta.json').read_text())
ci     = json.loads((data_dir / '_chunk_index.json').read_text())

# _chunk_index.json 값 형식: {"n": 토큰수, "sep": 위치}
total_tokens = sum(v["n"] for v in ci.values())

print(f'shards           : {len(shards)} 개')
print(f'총 토큰 수        : {total_tokens:,}')
print(f'vocab_size        : {meta["vocab_size"]}  (현재: {tok.vocab_size})')
print(f'fingerprint       : {meta["tokenizer_fingerprint"]}')
print(f'cond_tracks       : {meta["cond_tracks"]}')
print(f'target_tracks     : {meta["target_tracks"]}')

# fingerprint 검증 (불일치 시 SystemExit)
assert_data_matches_config(data_dir, tok)
print('\nfingerprint OK — 데이터와 설정이 일치합니다.')

## 4. 학습

`train.py`를 subprocess로 실행합니다. W&B는 비활성화하고 CSV 로거만 사용합니다.

예상 소요 시간 (T4 기준, bs=32, 1 epoch):

| 데이터 규모 | steps/epoch | 예상 시간 |
|---|---|---|
| 전체 (lakh+pop909+slakh, ~13k chunks) | ~400 | **~10~15 min** |
| POP909만 (~900 chunks) | ~28 | ~1 min |

> **loss 판정 기준:** train_loss가 초기(~5.0)보다 낮아지면 정상.  
> `torch.compile`은 기본값이 `false`이므로 T4/WSL2 환경에서도 별도 설정 없이 안전합니다.  
> 서버에서는 `--set model.compile=true`로 켜면 20–30% 빨라집니다.

In [ ]:
import os, subprocess, sys

env = os.environ.copy()
env['WANDB_DISABLED'] = 'true'

cmd = [
    sys.executable, 'scripts/train.py',
    '--epochs', str(EPOCHS),
    # ── Colab / T4 overrides ───────────────────────────────────────────────
    '--set', f'training.log_every_n_steps={LOG_EVERY}',
    '--set', 'training.warmup_steps=100',           # 5 epoch → step 수 적음
    '--set', 'training.checkpoint_every_n_train_steps=500',  # 드물게 저장
    '--set', 'training.early_stopping_enabled=false',  # 5 epoch에서는 ES 불필요
    # model.compile=false 가 config 기본값이므로 별도 override 불필요
]

print('실행 명령:', ' '.join(cmd))
print('-' * 60)

proc = subprocess.Popen(
    cmd, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
try:
    for line in proc.stdout:
        print(line, end='', flush=True)
finally:
    proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f'train.py 종료 코드: {proc.returncode}')
print('\n학습 완료!')

## 5-1. Loss 곡선 시각화

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# CSVLogger 가 쓴 metrics.csv 찾기
csv_files = sorted(glob.glob('logs/**/*.csv', recursive=True))
if not csv_files:
    print('metrics.csv 를 찾을 수 없습니다. logs/ 디렉토리를 확인하세요.')
else:
    df = pd.read_csv(csv_files[-1])
    print('CSV columns:', df.columns.tolist())
    print(df.tail(10).to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # ── Train loss (step 단위)
    if 'train_loss' in df.columns:
        tr = df[['step', 'train_loss']].dropna()
        axes[0].plot(tr['step'], tr['train_loss'], color='steelblue',
                     linewidth=0.8, label='train_loss')
        axes[0].set_title('Train Loss (per step)')
        axes[0].set_xlabel('step')
        axes[0].set_ylabel('CE loss')
        axes[0].yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
        axes[0].legend()
        axes[0].grid(alpha=0.3)

    # ── Val loss (epoch 단위)
    if 'val_loss' in df.columns:
        vl = df[['epoch', 'val_loss']].dropna()
        axes[1].plot(vl['epoch'], vl['val_loss'], 'o-', color='coral',
                     linewidth=1.5, markersize=6, label='val_loss')
        axes[1].set_title('Validation Loss (per epoch)')
        axes[1].set_xlabel('epoch')
        axes[1].set_ylabel('CE loss')
        axes[1].yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
        axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
        axes[1].legend()
        axes[1].grid(alpha=0.3)

        # 판정
        vl_vals = vl['val_loss'].tolist()
        decreasing = vl_vals[-1] < vl_vals[0]
        delta = vl_vals[0] - vl_vals[-1]
        status = 'OK: 하강 중' if decreasing else 'WARNING: 하강 없음'
        print(f'\nval_loss: {vl_vals[0]:.4f} -> {vl_vals[-1]:.4f}  '
              f'(delta={delta:+.4f})  [{status}]')

    plt.tight_layout()
    plt.savefig('output/loss_curve.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('loss_curve.png 저장 완료')

## 6. 추론 — 실제 POP909 멜로디 → 반주 생성

`data/processed/` 의 첫 번째 shard에서 melody 트랙을 추출해 프롬프트로 사용합니다.

> 학습 초기(5 epoch)라 생성 품질은 좋지 않습니다.  
> 중요한 것은 **파이프라인 전체가 오류 없이 동작하는지** 입니다.

In [ ]:
import torch
from pathlib import Path
from jam_transformer.config import load_config
from jam_transformer.tokenizer import build_tokenizer
from jam_transformer.midi_io import events_to_midi

cfg = load_config('configs/config.yaml')
tok = build_tokenizer(cfg.tokenizer)

# 첫 번째 shard 에서 melody 트랙만 추출
shards = sorted(p for p in Path('data/processed').glob('*.pt')
                if not p.name.startswith('_'))
data = torch.load(shards[0], map_location='cpu', weights_only=False)
ids  = data['ids'].tolist()

# TEMPO 복원
tempo = 120.0
for tid in ids:
    name = tok.id_to_token[tid] if 0 <= tid < tok.vocab_size else ''
    if name.startswith('TEMPO_'):
        tempo = tok.tempo_from_bin(int(name.split('_')[1]))
        break

# 전체 디코딩 → melody 만 남김
all_events = tok.decode(ids)
melody_events = [e for e in all_events if e.track == 'melody']
print(f'shard : {shards[0].name}')
print(f'tempo : {tempo:.1f} BPM')
print(f'melody notes : {len(melody_events)}')

# MIDI 저장
Path('output').mkdir(exist_ok=True)
melody_midi = Path('output/melody_prompt.mid')
events_to_midi(melody_events, cfg.tokenizer, tempo_bpm=tempo).dump(str(melody_midi))
print(f'\n저장 완료: {melody_midi}')

In [ ]:
import glob, os, sys

# 가장 최근 체크포인트 선택
ckpts = sorted(glob.glob('checkpoints/**/*.ckpt', recursive=True),
               key=os.path.getmtime)
if not ckpts:
    raise FileNotFoundError('체크포인트가 없습니다. 학습 셀을 먼저 실행하세요.')

ckpt = ckpts[-1]
print(f'사용할 체크포인트: {ckpt}')

cmd = [
    sys.executable, 'scripts/inference.py',
    '--checkpoint', ckpt,
    '--melody_midi', 'output/melody_prompt.mid',
    '--output',      'output/accompaniment.mid',
    '--temperature', '1.0',
    '--top_k',       '64',
    '--top_p',       '0.95',
    '--max_new_tokens', '512',
]
print('실행:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('inference.py 실패')
print('추론 완료!')

In [ ]:
import miditoolkit
from pathlib import Path

out_mid = 'output/accompaniment.mid'
m = miditoolkit.MidiFile(out_mid)

print(f'출력 MIDI: {out_mid}')
print(f'tempo    : {m.tempo_changes[0].tempo if m.tempo_changes else "(unknown)"}')
total_notes = 0
for inst in m.instruments:
    n = len(inst.notes)
    total_notes += n
    if n > 0:
        pitches = [note.pitch for note in inst.notes]
        print(f'  [{inst.name}]  {n} notes  '
              f'pitch={min(pitches)}-{max(pitches)}  '
              f'first5={[note.pitch for note in inst.notes[:5]]}')
print(f'\n총 생성 노트 수: {total_notes}')
if total_notes == 0:
    print('WARNING: 노트가 없습니다. max_new_tokens 를 늘리거나 '
          'temperature 를 높여 보세요.')
else:
    print('추론 파이프라인 정상 동작 확인!')

## 6-1. (선택) CFG 추론 비교

`cfg_w=0` (일반)과 `cfg_w=2.0` (CFG 강화)을 비교합니다.  
학습 초기에는 차이가 작을 수 있습니다.

In [ ]:
for cfg_w, label in [(0.0, 'nocfg'), (2.0, 'cfg2')]:
    out_path = f'output/accompaniment_{label}.mid'
    cmd = [
        sys.executable, 'scripts/inference.py',
        '--checkpoint', ckpt,
        '--melody_midi', 'output/melody_prompt.mid',
        '--output',      out_path,
        '--temperature', '1.0',
        '--top_k',       '64',
        '--max_new_tokens', '512',
        '--cfg_w',       str(cfg_w),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'cfg_w={cfg_w} 실패:', result.stderr[-300:])
        continue

    m = miditoolkit.MidiFile(out_path)
    notes = sum(len(i.notes) for i in m.instruments)
    print(f'cfg_w={cfg_w:4.1f}  ->  {notes} notes  [{out_path}]')

## 7. 체크포인트 + 결과물을 Drive에 저장

런타임이 종료되면 Colab 로컬 파일은 사라집니다.  
검증용 체크포인트와 loss 곡선을 Drive에 백업합니다.

In [ ]:
import shutil, glob
from pathlib import Path

# Drive 저장 경로
DRIVE_OUT = Path('/content/drive/MyDrive/jam_tx_colab_run')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

saved = []

# loss curve PNG
if Path('output/loss_curve.png').exists():
    dst = DRIVE_OUT / 'loss_curve.png'
    shutil.copy('output/loss_curve.png', dst)
    saved.append(str(dst))

# metrics.csv
for csv in glob.glob('logs/**/*.csv', recursive=True):
    dst = DRIVE_OUT / Path(csv).name
    shutil.copy(csv, dst)
    saved.append(str(dst))

# 생성된 MIDI
for mid in glob.glob('output/*.mid'):
    dst = DRIVE_OUT / Path(mid).name
    shutil.copy(mid, dst)
    saved.append(str(dst))

# last.ckpt
for ckpt_file in glob.glob('checkpoints/**/*.ckpt', recursive=True):
    dst = DRIVE_OUT / Path(ckpt_file).name
    shutil.copy(ckpt_file, dst)
    saved.append(str(dst))

print(f'Drive 저장 완료 ({len(saved)} 파일):')
for f in saved:
    print(' ', f)

## 요약 체크리스트

| 항목 | 확인 기준 |
|---|---|
| 데이터 검증 | fingerprint OK, shard 수 확인 |
| Train loss | 1 epoch 종료 시 초기보다 낮아졌으면 정상 |
| 추론 | accompaniment.mid 에 노트 > 0 |
| Drive 저장 | loss_curve.png + *.ckpt 백업 완료 |

---

**모두 통과했으면 4070 Ti Super 또는 서버에서 풀 학습으로 진행하세요!**

```powershell
# Windows + Docker Desktop (4070 Ti Super)
docker compose run --rm train python scripts/train.py

# native Linux 서버 (torch.compile 활성화)
docker compose run --rm train python scripts/train.py --set model.compile=true
```